## Steps of Feature Engineering
- Feature engineering starter 
- Target and Features Split
- Train Test Split
- Missing values handling
- Outliers detection and handing
- Categorical feature encoding
- Scaling
- Feature creation

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('dataset/titanic_data_updated.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

### Remove unnecessary columns

In [ ]:
df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
# df = df.drop(['Cabin'], axis=1)
df.sample(5)

### Separation of Target and Features

In [ ]:
X = df.drop('Survived', axis=1)
display(X.head())
y = df['Survived']
display(y.head())

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('X_train:', X_train.shape, '\nX_test:', X_test.shape)
print('y_train:', y_train.shape, '\ny_test:', y_test.shape)

## Imputing missing values

### Numerical missing value imputation using pandas

In [ ]:
X_train.head()

In [ ]:
df.isnull().sum()

In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
age_mean = X_train['Age'].mean()
age_median = X_train['Age'].median()
print('age_mean', age_mean)
print('age_median', age_median)

X_train['age_mean_imputer'] = X_train['Age'].fillna(age_mean) # imput only for X_train data
X_train['age_median_imputer'] = X_train['Age'].fillna(age_median)

X_train.info()

In [ ]:
import seaborn as sns

sns.kdeplot(X_train[['Age', 'age_mean_imputer', 'age_median_imputer']], fill=True)
# sns.kdeplot(X_train['age_mean_imputer'])

### Missing value imputation using sklearn

In [ ]:
from sklearn.impute import SimpleImputer

simpleImputer = SimpleImputer(missing_values=np.nan, strategy='mean')
simpleImputer

In [ ]:
simpleImputer.fit(X_train[['Age']])

In [ ]:
X_train['Age'] = simpleImputer.transform(X_train[['Age']])

In [ ]:
X_train.isnull().sum()

In [ ]:
X_train = X_train.drop(['age_mean_imputer', 'age_median_imputer'], axis=1)
X_train.columns

In [ ]:
age_median

In [ ]:
X_test.isnull().sum()

In [ ]:
X_test['Age'] = simpleImputer.transform(X_test[['Age']])
X_test.isnull().sum()

### Imputing missing values for categorical feature

In [ ]:
X_train['Embarked'].unique()

In [ ]:
print(X_train['Embarked'].isnull().sum())
print(X_test['Embarked'].isnull().sum())

In [ ]:
sns.countplot(X_train, x='Embarked')

In [ ]:
sns.countplot(X_test, x='Embarked')

In [ ]:
missing_embarked_indices = X_train.loc[X_train['Embarked'].isna()].index
X_train.loc[missing_embarked_indices]

In [ ]:
embarked_imputer = SimpleImputer(strategy='most_frequent')
embarked_imputer.fit(X_train[['Embarked']])

In [ ]:
X_train[['Embarked']] = embarked_imputer.transform(X_train[['Embarked']]) # ? should i use ravel() or not 
X_test[['Embarked']] = embarked_imputer.transform(X_test[['Embarked']])

X_train.loc[missing_embarked_indices]

### Categorical value imputation with 'missing' string and Indicator

In [ ]:
X_train['Cabin'].isnull().sum(), X_test['Cabin'].isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
sns.countplot(X_train, x='Cabin')
plt.xticks(rotation=90)
plt.show()

In [ ]:
cabin_imputer = SimpleImputer(
    missing_values = np.nan,
    strategy = 'constant',
    fill_value = 'missing',
    add_indicator = True
)

cabin_imputer.fit(X_train[['Cabin']])

In [ ]:
X_train[['Cabin', 'cabin_missing_indicator']] = cabin_imputer.transform(X_train[['Cabin']])

In [ ]:
X_test[['Cabin', 'cabin_missing_indicator']] = cabin_imputer.transform(X_test[['Cabin']])

In [ ]:
X_train.head()

In [ ]:
X_train.isnull().sum()

## Identify Data Distribution

In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(12, 5))

normal_data = np.random.normal(50, 10, 500)
stats.probplot(normal_data, plot=axes[0][0])
axes[0][0].set_title('QQ Plot - Normal Data')
sns.kdeplot(normal_data, ax=axes[0][1])
axes[0][1].set_title('KDE Plot - Normal Data')

skewed_data = np.random.exponential(10, 500)
stats.probplot(skewed_data, plot=axes[1][0])
axes[1][0].set_title('Skewed data')
sns.kdeplot(skewed_data, ax=axes[1][1])
axes[1][1].set_title('Skewed data')
plt.tight_layout()
plt.show()

In [ ]:


fig, axes = plt.subplots(2, 2, figsize=(12, 5))

stats.probplot(X_train['Age'], plot=axes[0][0])
axes[0][0].set_title('QQ Plot - Age (normal)')
sns.kdeplot(X_train['Age'], ax=axes[0][1])
axes[0][1].set_title('KDE Plot - Age')

stats.probplot(X_train['Fare'], plot=axes[1][0])
axes[1][0].set_title('Fare (Skewed data)')
sns.kdeplot(X_train['Fare'], ax=axes[1][1])
axes[1][1].set_title('Skewed data')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Create datasets with different shapes
normal_data = np.random.normal(loc=50, scale=10, size=500)
right_skewed = np.random.exponential(scale=10, size=500)
left_skewed = 100 - np.random.exponential(scale=10, size=500)
heavy_tailed = stats.t.rvs(df=3, loc=50, scale=10, size=500)

datasets = {
    "Normal": normal_data,
    "Right Skewed": right_skewed,
    "Left Skewed": left_skewed,
    "Heavy Tailed": heavy_tailed,
}

### Visual Tests

### Histogram + KDE overlay (simplest first check)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (name, data) in zip(axes.flatten(), datasets.items()):
    sns.histplot(data, bins=30, kde=True, stat="density", ax=ax)
    ax.set_title(name)

plt.tight_layout()
plt.show()

### KDE Plot comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All on one plot
for name, data in datasets.items():
    sns.kdeplot(data=data, label=name, ax=axes[0], fill=True, alpha=0.2)
axes[0].set_title("All Distributions Overlaid")
axes[0].legend()

# Normal curve reference
x = np.linspace(0, 100, 500)
axes[1].plot(x, stats.norm.pdf(x, 50, 10), "k--", linewidth=2, label="Perfect Normal")
sns.kdeplot(data=normal_data, ax=axes[1], label="My Data")
axes[1].set_title("My Data vs Perfect Normal")
axes[1].legend()

plt.tight_layout()
plt.show()

### QQ Plot
Best visual test. Most reliable

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, data) in zip(axes.flatten(), datasets.items()):
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"QQ Plot — {name}")
    ax.get_lines()[0].set_markerfacecolor("steelblue")  # data points
    ax.get_lines()[0].set_markersize(4)
    ax.get_lines()[1].set_color("red")                  # reference line

plt.tight_layout()
plt.show()


Detailed reading QQ Plot:

Pattern 	            Meaning
- Points hug the line -> Normally distributed
- Points curve up at right end ->	Right-skewed (heavy right tail)
- Points curve down at left end ->	Left-skewed (heavy left tail)
- S-shape (steep ends) ->	Heavy tails (leptokurtic)
- Inverse S-shape (flat ends) ->	Light tails (platykurtic)

### Boxplot

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (name, data) in zip(axes, datasets.items()):
    sns.boxplot(data=data, ax=ax, color="steelblue", orient="v")
    ax.set_title(name)

plt.tight_layout()
plt.show()

Boxplot reading:
- Normal : Median centered, Equal whiskers        
- Right Skewed: Median near Q1, Right whisker long
- Left Skewed: Median near Q3, Left whisker long

### Violine Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

all_data = []
labels = []
for name, data in datasets.items():
    all_data.extend(data)
    labels.extend([name] * len(data))

df = pd.DataFrame({"value": all_data, "distribution": labels})
sns.violinplot(data=df, x="distribution", y="value", palette="Set2", inner="box", ax=ax)
ax.set_title("Violin Plot — Shape Comparison")
plt.show()

### Statistical Tests

### Skewness value

In [ ]:
for name, data in datasets.items():
    skew = stats.skew(data)
    print(f"{name:15s}  Skewness = {skew:+.3f}")

Interpretation:


Skewness Value	Meaning
- 0 ->	Perfectly symmetric
- -0.5 to +0.5 ->	Approximately normal
- -1 to -0.5 ->	Moderately left-skewed
- +0.5 to +1 ->	Moderately right-skewed
- < -1 ->	Highly left-skewed
- \> +1 ->	Highly right-skewed

### Kurtosis Value

In [ ]:
for name, data in datasets.items():
    kurt = stats.kurtosis(data)  # excess kurtosis (normal = 0)
    print(f"{name:15s}  Kurtosis = {kurt:+.3f}")

| Kurtosis	| Meaning |
| --- | --- |
| 0 |	Normal tails (mesokurtic) |
| > 0 |	Heavy tails (leptokurtic) — more outliers |
| < 0 |	Light tails (platykurtic) — fewer outliers |

### Shapiro Wilk Test (best for small-medium data)

In [ ]:
for name, data in datasets.items():
    stat, p_value = stats.shapiro(data)
    normal = "YES" if p_value > 0.05 else "NO"
    print(f"{name:15s}  W={stat:.4f}  p={p_value:.6f}  Normal? {normal}")

Interpretation:

- p > 0.05 → Fail to reject normality → Data could be normal
- p ≤ 0.05 → Reject normality → Data is not normal

### D'Agostino-Pearson Test (Good for Larger Data)

In [ ]:
for name, data in datasets.items():
    stat, p_value = stats.normaltest(data)
    normal = "YES" if p_value > 0.05 else "NO"
    print(f"{name:15s}  stat={stat:.4f}  p={p_value:.6f}  Normal? {normal}")

### Complete Diagnostic Function

In [ ]:
def diagnose_distribution(data, name="Data"):
    """Full normality and skewness diagnosis."""

    print(f"{'='*55}")
    print(f" DIAGNOSIS: {name}")
    print(f"{'='*55}")

    # ── Basic Stats ──
    n = len(data)
    mean = np.mean(data)
    median = np.median(data)
    std = np.std(data)
    skew_val = stats.skew(data)
    kurt_val = stats.kurtosis(data)

    print(f"\n--- Basic Statistics ---")
    print(f"  n       = {n}")
    print(f"  Mean    = {mean:.3f}")
    print(f"  Median  = {median:.3f}")
    print(f"  Std     = {std:.3f}")
    print(f"  Skew    = {skew_val:+.3f}", end="")
    if abs(skew_val) < 0.5:
        print("  (≈ symmetric)")
    elif abs(skew_val) < 1:
        print("  (moderate skew)")
    else:
        print("  (high skew!)")

    print(f"  Kurt    = {kurt_val:+.3f}", end="")
    if abs(kurt_val) < 1:
        print("  (≈ normal tails)")
    else:
        print("  (deviates from normal tails)")

    # ── Mean vs Median check ──
    diff_pct = abs(mean - median) / std * 100
    print(f"\n  Mean-Median difference: {abs(mean-median):.3f} ({diff_pct:.1f}% of std)")
    if diff_pct < 5:
        print("  → Mean ≈ Median → likely symmetric")
    elif mean > median:
        print("  → Mean > Median → likely right-skewed")
    else:
        print("  → Mean < Median → likely left-skewed")

    # ── Statistical Tests ──
    print(f"\n--- Normality Tests ---")

    if n >= 8 and n <= 5000:
        sw_stat, sw_p = stats.shapiro(data)
        print(f"  Shapiro-Wilk:    W={sw_stat:.4f}  p={sw_p:.6f}  {'NORMAL' if sw_p > 0.05 else 'NOT NORMAL'}")

    if n >= 20:
        dp_stat, dp_p = stats.normaltest(data)
        print(f"  D'Agostino:      stat={dp_stat:.4f}  p={dp_p:.6f}  {'NORMAL' if dp_p > 0.05 else 'NOT NORMAL'}")

    ks_stat, ks_p = stats.kstest(data, "norm", args=(mean, std))
    print(f"  Kolmogorov-Smirnov: D={ks_stat:.4f}  p={ks_p:.6f}  {'NORMAL' if ks_p > 0.05 else 'NOT NORMAL'}")

    # ── Visual ──
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Histogram + KDE
    sns.histplot(data, bins=30, kde=True, stat="density", ax=axes[0], color="steelblue")
    x_range = np.linspace(data.min(), data.max(), 200)
    axes[0].plot(x_range, stats.norm.pdf(x_range, mean, std), "r--", linewidth=2, label="Normal fit")
    axes[0].set_title(f"Histogram — {name}")
    axes[0].legend()

    # QQ Plot
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].get_lines()[0].set_markerfacecolor("steelblue")
    axes[1].get_lines()[0].set_markersize(4)
    axes[1].get_lines()[1].set_color("red")
    axes[1].set_title(f"QQ Plot — {name}")

    # Boxplot
    sns.boxplot(data=data, ax=axes[2], color="steelblue", orient="v")
    axes[2].set_title(f"Boxplot — {name}")

    plt.tight_layout()
    plt.show()

    # ── Verdict ──
    print(f"\n--- Verdict ---")
    normal_tests_passed = 0
    total_tests = 0
    if n >= 8 and n <= 5000:
        total_tests += 1
        if sw_p > 0.05: normal_tests_passed += 1
    if n >= 20:
        total_tests += 1
        if dp_p > 0.05: normal_tests_passed += 1
    total_tests += 1
    if ks_p > 0.05: normal_tests_passed += 1

    if normal_tests_passed == total_tests:
        print("  ✓ Data appears NORMALLY DISTRIBUTED")
        print("  → Safe to use parametric tests (t-test, ANOVA, Pearson correlation)")
    elif normal_tests_passed >= total_tests / 2:
        print("  ~ Data is APPROXIMATELY normal (borderline)")
        print("  → Consider both parametric and non-parametric tests")
    else:
        print("  ✗ Data is NOT normally distributed")
        if skew_val > 0.5:
            print(f"  → Right-skewed (skewness={skew_val:+.3f})")
            print("  → Try: log transform, sqrt transform, or non-parametric tests")
        elif skew_val < -0.5:
            print(f"  → Left-skewed (skewness={skew_val:+.3f})")
            print("  → Try: square transform, reflect+log, or non-parametric tests")
        else:
            print(f"  → Non-normal but roughly symmetric (kurtosis issue)")
            print("  → Try: non-parametric tests (Mann-Whitney, Kruskal-Wallis)")

In [ ]:
diagnose_distribution(normal_data, "Normal Data")
diagnose_distribution(right_skewed, "Right Skewed Data")

## Outlier Detection

### Z-Score/Standardization Method

- Better for normally distributed data
- make use of population mean and std
- Sensitive to outliers as mean and std varies with outliers

In [ ]:
X_train.head()

In [ ]:
sns.kdeplot(X_train, x='Age', fill=True)

In [ ]:
age_mean = X_train['Age'].mean()
age_std = X_train['Age'].std()
age_mean, age_std

In [ ]:
X_train['age_z_score'] = (X_train['Age'] - age_mean) / age_std
X_train.head()

In [ ]:
sns.boxplot(X_train['age_z_score'])
X_train['age_z_score'].describe()

In [ ]:
age_outliers_zs = X_train[abs(X_train['age_z_score']) > 3]
age_outliers_zs

In [ ]:
sns.kdeplot(X_train['Fare'], fill=True)

In [ ]:
print(X_train['Fare'].describe())
sns.boxplot(X_train['Fare'])

In [ ]:
fare_mean = X_train['Fare'].mean()
fare_std = X_train['Fare'].std()

X_train['fare_zscore'] = (X_train['Fare'] - fare_mean) / fare_std
X_train.sample(5)

In [ ]:
X_train['fare_zscore'].describe()

In [ ]:
fare_outliers_zs = X_train[abs(X_train['fare_zscore']) > 3]
fare_outliers_zs.head()

### IQR Method

- Better for any kinds of data
- Based on median of population
- Robust to outliers as median is not sensitive to outliers

In [ ]:
sns.kdeplot(X_train['Age'])

In [ ]:
X_train['Age'].describe()

In [ ]:
age_Q1 = X_train['Age'].quantile(0.25)
age_Q3 = X_train['Age'].quantile(0.75)
print('quantiles:', age_Q1, age_Q3)

age_IQR = age_Q3 - age_Q1
print('IQR:', age_IQR)

age_min = age_Q1 - 1.5*age_IQR
age_max = age_Q3 + 1.5*age_IQR
print('min-max:', age_min, age_max)

In [ ]:
sns.boxplot(X_train['Age'])

In [ ]:
age_outliers_iqr = X_train[(X_train['Age'] < age_min) | (X_train['Age'] > age_max)]
print(age_outliers_iqr.shape)
age_outliers_iqr.sample(5)

In [ ]:
sns.kdeplot(X_train['Fare'])

In [ ]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
print('q1, q3:', fare_Q1, fare_Q3)

fare_IQR = fare_Q3 - fare_Q1
print('fare IQR:', fare_IQR)

fare_min = fare_Q1 - fare_IQR * 1.5
fare_max = fare_Q3 + fare_IQR * 1.5
print('min-max:', fare_min, fare_max)

In [ ]:
sns.boxplot(X_train['Fare'])

In [ ]:
fare_outliers_iqr = X_train[(X_train['Fare'] < fare_min) | (X_train['Fare'] > fare_max)]
print(fare_outliers_iqr.shape)
fare_outliers_iqr

### Winsorization/Percentile Technique

- Rarely used

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('dataset/marks_dataset.csv')
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15,5))
sns.kdeplot(df['maths_marks'], ax=ax[0])
sns.boxplot(df['maths_marks'], ax=ax[1])
plt.tight_layout()

In [ ]:
df['maths_marks'].describe()

#### Selecting Percentile value

In [ ]:
print(f'{"Percentile":<10}{"Min":^10}{"Max":>10}')

for p in [1, 2, 3, 5, 7, 8, 9, 10, 11, 12, 13, 14]:
    min = df['maths_marks'].quantile(p/100)
    max = df['maths_marks'].quantile(1-p/100)

    print(f'{p:<10}{min:^10.2f}{max:>10.2f}')

#### Capping

In [ ]:
x = 0.05

min_range = df['maths_marks'].quantile(x)
max_range = df['maths_marks'].quantile(1-x)

print('min-max range:', min_range, max_range)

df['maths_marks'] = df['maths_marks'].clip(min_range, max_range)
df['maths_marks'].describe()

## Outlier Handling

### Outlier Handling (Removing data point)

In [ ]:
print(age_outliers_zs.shape)
print(X_train.shape)
X_train = X_train[abs(X_train['age_z_score']) <= 3]
print(X_train.shape)

### Outlier Handling (Capping data point)

In [ ]:
print(fare_outliers_iqr.shape)
fare_min = max(0, fare_Q1 - fare_IQR * 1.5)
fare_max = fare_Q3 + fare_IQR * 1.5
print('fare min-max:', fare_min, fare_max)

X_train[X_train['Fare'] > fare_max].head()

In [ ]:
X_train['Fare'] = X_train['Fare'].clip(fare_min, fare_max)
X_train[X_train['Fare'] == fare_max].head()

## Encoding

### Ordinal Encoding with OrdinalEncoder

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X_train.head()

In [ ]:
pclass_enc = OrdinalEncoder(categories=[['third', 'second', 'first']])

In [ ]:
pclass_enc.fit(X_train[['Pclass']])
pclass_enc.categories_

In [ ]:
print(pclass_enc.transform(X_train[['Pclass']]).shape)
pclass_enc.transform(X_train[['Pclass']])[:10].T

In [ ]:
X_train['pclass_enc'] = pclass_enc.transform(X_train[['Pclass']])
X_test['pclass_enc'] = pclass_enc.transform(X_test[['Pclass']])

X_train.head()

In [ ]:
X_test.head()

In [ ]:
X_train.drop('Pclass', axis=1, inplace=True)
X_test.drop('Pclass', axis=1, inplace=True)

display(X_train.info())
display(X_test.info())

### Nominal Encoding with OneHotEncoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder

sex_ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
sex_ohe.fit(X_train[['Sex']])
sex_ohe.categories_

In [ ]:
sex_enc_train = sex_ohe.transform(X_train[['Sex']])
sex_enc_test = sex_ohe.transform(X_test[['Sex']])

sex_enc_train.head()

In [ ]:
X_train = pd.concat([X_train, sex_enc_train], axis=1)
X_test = pd.concat([X_test, sex_enc_test], axis=1)
X_train.head()

In [ ]:
embarked_ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
embarked_ohe.fit(X_train[['Embarked']])
print(embarked_ohe.categories_)

embarked_enc_train = embarked_ohe.transform(X_train[['Embarked']])
embarked_enc_test = embarked_ohe.transform(X_test[['Embarked']])

X_train = pd.concat([X_train, embarked_enc_train], axis=1)
X_test = pd.concat([X_test, embarked_enc_test], axis=1)

X_train.head()
X_test.head()


### Removing original columns

In [ ]:
X_train = X_train.drop(['Sex', 'Embarked'], axis=1)
X_test = X_test.drop(['Sex', 'Embarked'], axis=1)

X_train.head()

### Encoding Cabin Feature

In [ ]:
X_train['Cabin'].value_counts()

In [ ]:
X_train['cabin_deck'] = X_train['Cabin'].str[0]
X_train['cabin_deck'].value_counts()

In [ ]:
X_test['Cabin'].value_counts()

In [ ]:
X_test['cabin_deck'] = X_test['Cabin'].str[0]
X_test['cabin_deck'].value_counts()

In [ ]:
cabin_ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
cabin_ohe.fit(X_train[['cabin_deck']])
cabin_ohe.categories_

In [ ]:
cabin_enc_train = cabin_ohe.transform(X_train[['cabin_deck']])
cabin_enc_test = cabin_ohe.transform(X_test[['cabin_deck']])

X_train = pd.concat([X_train, cabin_enc_train], axis=1)
X_test = pd.concat([X_test, cabin_enc_test], axis=1)

X_train = X_train.drop('Cabin', axis=1)
X_test = X_test.drop('Cabin', axis=1)

X_train.info()

### Label Encoding

In [ ]:
y_train.info()

In [ ]:
y_train.sample(5)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)
y_train_arr = le.transform(y_train)
y_test_arr = le.transform(y_test)

print(type(y_test_arr))
y_test_arr

In [ ]:
y_test_enc = pd.Series(y_test_arr, index=y_test.index, name=y_test.name)
y_test_enc

In [ ]:
y_train_enc = pd.Series(y_train_arr, index=y_train.index, name=y_train.name)


## Scaling

### Standard Scaling

In [ ]:
X_train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(X_train[['Age']])
print('scale:', ss.scale_)
print('n_features:', ss.n_features_in_)
print('mean:', ss.mean_)

In [ ]:
X_train['age_ss'] = ss.transform(X_train[['Age']])
X_test['age_ss'] = ss.transform(X_test[['Age']])
X_train.head()

In [ ]:
X_train['age_ss'].describe() # mean = 0, std = 1

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,5))

sns.kdeplot(X_train['Age'], ax=ax[0])
ax[0].set_title('Age Distribution')

sns.kdeplot(X_train['age_ss'], ax=ax[1])
ax[1].set_title('age standard scaled (distribution unchanged)')

### Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X_train.head()

In [ ]:
mms = MinMaxScaler()
mms.fit(X_train[['Fare']])

In [ ]:
mms.scale_, mms.data_min_

In [ ]:
X_train['fare_mms'] = mms.transform(X_train[['Fare']])
X_test['fare_mms'] = mms.transform(X_test[['Fare']])
X_train.head()

In [ ]:
X_train['fare_mms'].describe()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(8, 4))
sns.kdeplot(X_train['Fare'], ax=ax[0])
ax[0].set_title('Fare')
sns.kdeplot(X_train['fare_mms'], ax=ax[1])
ax[1].set_title('fare_mms')